Exploratory Data Analysis

Understanding the dataset to explore how the data is present in the database and if there is a need of creating some aggregated tables that can help with:
    Vendor selection for profitability
    Product Pricing Optimization

In [14]:
import pandas as pd
import sqlite3
import sqlite3
import os
import time
import logging
from sqlalchemy import create_engine


In [16]:
# ==========================================
# PART 1: EXPLORING THE RAW DATABASE 
# (From transcript 00:21:24 to 00:34:30)
# ==========================================
print("--- PART 1: Exploring Raw Tables ---")

# Create the database connection
conn = sqlite3.connect('inventory.db')

# Check which tables are present
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql_query(query_tables, conn)
print("Tables in database:")
print(tables_df)
tables = tables_df['name'].tolist()

# Print row counts and top 5 rows for each table
for table in tables:
    print(f"\n{table} " + "-" * 50)
    count_df = pd.read_sql_query(f"SELECT count(*) as count FROM {table}", conn)
    print(f"Count of records: {count_df['count'][0]}")
    
    data_df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5", conn)
    print(data_df.head())


# ==========================================
# PART 2: CREATING THE FINAL AGGREGATED SCRIPT 
# (From transcript 00:34:30 to 01:02:33)
# ==========================================
print("\n--- PART 2: Running the Aggregation Script ---")

# Ensure logs directory exists
os.makedirs('logs', exist_ok=True)

# Configure Logging
logging.basicConfig(
    filename='logs/get_vendor_summary.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

def create_vendor_summary(conn):
    """Executes heavy SQL joins to create the summarized table."""
    logging.info("Executing heavy SQL join query to summarize vendors...")
    
    # We combine the freight summary, purchase summary, and sales summary using SQL CTEs (WITH clauses)
    query = """
    WITH FreightSummary AS (
        SELECT VendorNumber, SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),
    PurchaseSummary AS (
        SELECT 
            p.VendorNumber, 
            p.VendorName, 
            p.Brand, 
            p.Description,
            pp.Price AS ActualPrice, 
            p.PurchasePrice, 
            SUM(p.Quantity) AS TotalPurchaseQuantity, 
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchases p
        LEFT JOIN purchase_prices pp ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description
    ),
    SalesSummary AS (
        SELECT 
            VendorNumber, 
            Brand, 
            SUM(SalesQuantity) AS TotalSalesQuantity, 
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNumber, Brand
    )
    SELECT 
        ps.VendorNumber, 
        ps.VendorName, 
        ps.Brand, 
        ps.Description,
        ps.PurchasePrice, 
        ps.ActualPrice, 
        ss.TotalSalesQuantity, 
        ss.TotalSalesDollars, 
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        ps.TotalPurchaseQuantity, 
        ps.TotalPurchaseDollars, 
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss ON ps.VendorNumber = ss.VendorNumber AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs ON ps.VendorNumber = fs.VendorNumber;
    """
    
    df = pd.read_sql_query(query, conn)
    return df

def clean_data(df):
    """Cleans data and creates new calculated features."""
    logging.info("Cleaning data and adding new calculated features...")
    
    # Fill missing sales values with 0
    df.fillna(0, inplace=True)
    
    # Remove irrelevant white spaces from Vendor names
    df['VendorName'] = df['VendorName'].str.strip()
    
    # Create Gross Profit: Sales Dollars - Purchase Dollars
    df['Gross_Profit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    
    import numpy as np
    # Create Profit Margin %
    df['Profit_Margin'] = np.where(df['TotalSalesDollars'] > 0, 
                                   (df['Gross_Profit'] / df['TotalSalesDollars']) * 100, 0)
    
    # Create Stock Turnover
    df['Stock_Turnover'] = np.where(df['TotalPurchaseQuantity'] > 0,
                                    df['TotalSalesQuantity'] / df['TotalPurchaseQuantity'], 0)
    
    # Create Sales to Purchase Ratio
    df['Sales_to_Purchase_Ratio'] = np.where(df['TotalPurchaseDollars'] > 0,
                                             df['TotalSalesDollars'] / df['TotalPurchaseDollars'], 0)
    
    return df

if __name__ == "__main__":
    start_time = time.time()
    
    engine = create_engine('sqlite:///inventory.db')
    
    try:
        # Step 1: Run query to combine tables
        summary_df = create_vendor_summary(conn)
        logging.info(f"Summary table successfully created with {len(summary_df)} rows.")
        print(f"Aggregated {len(summary_df)} rows from raw tables.")
        
        # Step 2: Clean data and calculate KPIs
        cleaned_df = clean_data(summary_df)
        print("Data cleaned and KPIs calculated.")
        
        # Step 3: Ingest back to database
        logging.info("Ingesting 'vendor_sales_summary' back into database...")
        cleaned_df.to_sql('vendor_sales_summary', con=engine, if_exists='replace', index=False)
        print("Success! 'vendor_sales_summary' saved to database.")
        
        end_time = time.time()
        logging.info(f"Process completed successfully in {(end_time - start_time) / 60:.2f} minutes.")
        
    except Exception as e:
        logging.error(f"An error occurred: {e}")
        print(f"Error: {e}")
        
    finally:
        conn.close()

--- PART 1: Exploring Raw Tables ---
Tables in database:
              name
0  begin_inventory
1    end_inventory
2        purchases
3  purchase_prices
4            sales
5   vendor_invoice

begin_inventory --------------------------------------------------
Count of records: 206529
         InventoryId  Store          City  Brand                  Description  \
0  1_HARDERSFIELD_58      1  HARDERSFIELD     58  Gekkeikan Black & Gold Sake   
1  1_HARDERSFIELD_60      1  HARDERSFIELD     60       Canadian Club 1858 VAP   
2  1_HARDERSFIELD_62      1  HARDERSFIELD     62     Herradura Silver Tequila   
3  1_HARDERSFIELD_63      1  HARDERSFIELD     63   Herradura Reposado Tequila   
4  1_HARDERSFIELD_72      1  HARDERSFIELD     72         No. 3 London Dry Gin   

    Size  onHand  Price   startDate  
0  750mL       8  12.99  2024-01-01  
1  750mL       7  10.99  2024-01-01  
2  750mL       6  36.99  2024-01-01  
3  750mL       3  38.99  2024-01-01  
4  750mL       6  34.99  2024-01-01  

e

In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('inventory.db')

print("vendor_invoice columns:")
print(pd.read_sql("SELECT * FROM vendor_invoice LIMIT 0", conn).columns.tolist())

print("\npurchases columns:")
print(pd.read_sql("SELECT * FROM purchases LIMIT 0", conn).columns.tolist())

print("\nsales columns:")
print(pd.read_sql("SELECT * FROM sales LIMIT 0", conn).columns.tolist())

vendor_invoice columns:
['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate', 'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval']

purchases columns:
['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber', 'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate', 'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification']

sales columns:
['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity', 'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification', 'ExciseTax', 'VendorNo', 'VendorName']
